1) 그래프


![problem1-1](problem1.jpg)

In [1]:
#dfs

import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as mpatches
import networkx as nx

# ── 1. 그래프 데이터 ──────────────────────────────────────────────
edges = [
    ('소테',          '유림닭도리탕',  37),
    ('유림닭도리탕',  '맛있는 칼국수', 14),
    ('맛있는 칼국수', '오몬자(마포)',  11),
    ('오몬자(마포)',  '남산광어',      37),
    ('남산광어',      '와이인',        29),
    ('와이인',        '효뜨',          21),
    ('효뜨',          '문화식당',      36),
    ('문화식당',      '스페이스',      22),
    ('스페이스',      '르부아',        18),
    ('르부아',        '오몬자(강남)',  37),
    ('오몬자(강남)',  '라라면가',      39),
    ('라라면가',      '타코스퀘어',    25),
    ('타코스퀘어',    '미엔아이',      26),
    ('미엔아이',      '따로집',        39),
    ('따로집',        '코지밀',        42),
    ('코지밀',        '푸주옥',        36),
    ('푸주옥',        '비욘드비엣남',  19),
    ('비욘드비엣남',  '프레고클럽',    27),
    ('프레고클럽',    '이태원화로',    25),
    ('이태원화로',    '소테',          31),
]

pos = {
    '소테':          (0.50, 0.92),
    '유림닭도리탕':  (0.82, 0.83),
    '맛있는 칼국수': (0.94, 0.65),
    '오몬자(마포)':  (0.82, 0.45),
    '남산광어':      (0.63, 0.30),
    '와이인':        (0.38, 0.22),
    '효뜨':          (0.15, 0.35),
    '문화식당':      (0.07, 0.55),
    '스페이스':      (0.16, 0.75),
    '르부아':        (0.37, 0.90),
    '오몬자(강남)':  (0.57, 0.62),
    '라라면가':      (0.75, 0.62),
    '타코스퀘어':    (0.70, 0.78),
    '미엔아이':      (0.49, 0.76),
    '따로집':        (0.29, 0.62),
    '코지밀':        (0.22, 0.47),
    '푸주옥':        (0.44, 0.45),
    '비욘드비엣남':  (0.68, 0.17),
    '프레고클럽':    (0.90, 0.28),
    '이태원화로':    (0.93, 0.47),
}

G = nx.Graph()
for u, v, w in edges:
    G.add_edge(u, v, weight=w)

# ── 2. DFS 순서 계산 ──────────────────────────────────────────────
def dfs_steps(graph, start):
    visited = set()
    steps = []

    def dfs(node, from_node):
        visited.add(node)
        edge = (from_node, node) if from_node else None
        steps.append((frozenset(visited), edge, list(visited.__class__(visited))))
        # 방문 순서 따로 관리
        steps[-1] = (frozenset(visited), edge, order[:])
        for neighbor in sorted(graph.neighbors(node)):
            if neighbor not in visited:
                order.append(neighbor)
                dfs(neighbor, node)

    order = [start]
    dfs(start, None)
    return steps

start_node = '소테'
steps = dfs_steps(G, start_node)

# ── 3. 색상 ───────────────────────────────────────────────────────
COLOR_DEFAULT   = '#EEEDFE'
COLOR_VISITED   = '#534AB7'
COLOR_CURRENT   = '#EF9F27'
COLOR_EDGE_ON   = '#EF9F27'
COLOR_EDGE_DONE = '#534AB7'
COLOR_EDGE_OFF  = '#CCCCCC'
OMONJA_COLOR    = '#F0997B'

edge_list   = list(G.edges())
edge_weight = {(u,v): d['weight'] for u,v,d in G.edges(data=True)}
edge_weight.update({(v,u): d['weight'] for u,v,d in G.edges(data=True)})

# ── 4. Figure 생성 & 저장까지 한 셀에서 ──────────────────────────
fig, ax = plt.subplots(figsize=(11, 9))
fig.patch.set_facecolor('#F8F7FF')

def update(frame):
    ax.clear()
    ax.set_facecolor('#F8F7FF')
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.axis('off')

    visited_set, active_edge, order_list = steps[frame]
    current_node = order_list[-1]

    done_edges = set()
    for i in range(len(order_list) - 1):
        done_edges.add((order_list[i], order_list[i+1]))
        done_edges.add((order_list[i+1], order_list[i]))

    # 엣지
    for u, v in edge_list:
        x0, y0 = pos[u]; x1, y1 = pos[v]
        is_active = active_edge and {u,v} == {active_edge[0], active_edge[1]}
        is_done   = (u,v) in done_edges
        color = COLOR_EDGE_ON if is_active else (COLOR_EDGE_DONE if is_done else COLOR_EDGE_OFF)
        lw    = 3.5 if is_active else (2.2 if is_done else 1.0)
        ax.plot([x0,x1],[y0,y1], color=color, lw=lw, zorder=2, solid_capstyle='round')

        mx,my = (x0+x1)/2,(y0+y1)/2
        dx,dy = x1-x0,y1-y0
        ln = (dx**2+dy**2)**0.5 or 1
        ox,oy = -dy/ln*0.025, dx/ln*0.025
        fc = color if is_active else ('#888' if is_done else '#AAAAAA')
        ax.text(mx+ox, my+oy, str(edge_weight[(u,v)])+'분',
                fontsize=7.5, ha='center', va='center', color=fc,
                fontfamily='Malgun Gothic', zorder=4,
                bbox=dict(boxstyle='round,pad=0.15', fc='#F8F7FF', ec='none', alpha=0.85))

    # 노드
    for node in G.nodes():
        x,y = pos[node]
        is_cur  = node == current_node
        is_vis  = node in visited_set
        is_om   = node.startswith('오몬자')
        if is_cur:
            fc,ec,lw,r = COLOR_CURRENT,'#BA7517',2.5,0.052
        elif is_vis:
            fc,ec,lw,r = COLOR_VISITED,'#3C3489',1.5,0.046
        else:
            fc = OMONJA_COLOR if is_om else COLOR_DEFAULT
            ec = '#993C1D' if is_om else '#AFA9EC'
            lw,r = 1.0,0.046
        ax.add_patch(plt.Circle((x,y),r,fc=fc,ec=ec,lw=lw,zorder=5))
        tc = 'white' if (is_vis or is_cur) else ('#993C1D' if is_om else '#3C3489')
        ax.text(x,y,node,ha='center',va='center',fontsize=7.2,
                fontfamily='Malgun Gothic',color=tc,fontweight='bold',zorder=6)
        if is_vis:
            idx = order_list.index(node)+1
            ax.text(x+r*0.72,y+r*0.72,str(idx),fontsize=7,color='white',
                    fontweight='bold',ha='center',va='center',zorder=7,
                    fontfamily='Malgun Gothic',
                    bbox=dict(boxstyle='circle,pad=0.15',
                              fc=COLOR_CURRENT if is_cur else COLOR_VISITED,ec='none'))

    ax.text(0.5,1.03,f'DFS 탐색 순서  ({frame+1}/{len(steps)})',
            ha='center',va='center',fontsize=10,
            fontfamily='Malgun Gothic',color='#3C3489',fontweight='bold')
    ax.text(0.5,0.98,' → '.join(order_list),
            ha='center',va='center',fontsize=8,
            fontfamily='Malgun Gothic',color='#534AB7')
    ax.legend(handles=[
        mpatches.Patch(fc=COLOR_CURRENT,ec='#BA7517',label='현재 방문 중'),
        mpatches.Patch(fc=COLOR_VISITED,ec='#3C3489',label='방문 완료'),
        mpatches.Patch(fc=COLOR_DEFAULT,ec='#AFA9EC',label='미방문'),
    ], loc='lower right', prop={'family':'Malgun Gothic','size':8},
       framealpha=0.85, edgecolor='#CCCCCC')

ani = animation.FuncAnimation(fig, update, frames=len(steps), interval=1000, repeat=False)

# ── 5. mp4 저장 (창 열기 전에 저장) ──────────────────────────────
ani.save('dfs.mp4', writer='ffmpeg', fps=1, dpi=150)
print('dfs.mp4 저장 완료!')

# ── 6. 팝업 창으로 확인 ───────────────────────────────────────────
plt.tight_layout()
plt.show()

dfs.mp4 저장 완료!


In [2]:
#bfs

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as mpatches
import networkx as nx
from collections import deque

# ── 1. 그래프 데이터 ──────────────────────────────────────────────
edges = [
    ('소테',          '유림닭도리탕',  37),
    ('유림닭도리탕',  '맛있는 칼국수', 14),
    ('맛있는 칼국수', '오몬자(마포)',  11),
    ('오몬자(마포)',  '남산광어',      37),
    ('남산광어',      '와이인',        29),
    ('와이인',        '효뜨',          21),
    ('효뜨',          '문화식당',      36),
    ('문화식당',      '스페이스',      22),
    ('스페이스',      '르부아',        18),
    ('르부아',        '오몬자(강남)',  37),
    ('오몬자(강남)',  '라라면가',      39),
    ('라라면가',      '타코스퀘어',    25),
    ('타코스퀘어',    '미엔아이',      26),
    ('미엔아이',      '따로집',        39),
    ('따로집',        '코지밀',        42),
    ('코지밀',        '푸주옥',        36),
    ('푸주옥',        '비욘드비엣남',  19),
    ('비욘드비엣남',  '프레고클럽',    27),
    ('프레고클럽',    '이태원화로',    25),
    ('이태원화로',    '소테',          31),
]

pos = {
    '소테':          (0.50, 0.92),
    '유림닭도리탕':  (0.82, 0.83),
    '맛있는 칼국수': (0.94, 0.65),
    '오몬자(마포)':  (0.82, 0.45),
    '남산광어':      (0.63, 0.30),
    '와이인':        (0.38, 0.22),
    '효뜨':          (0.15, 0.35),
    '문화식당':      (0.07, 0.55),
    '스페이스':      (0.16, 0.75),
    '르부아':        (0.37, 0.90),
    '오몬자(강남)':  (0.57, 0.62),
    '라라면가':      (0.75, 0.62),
    '타코스퀘어':    (0.70, 0.78),
    '미엔아이':      (0.49, 0.76),
    '따로집':        (0.29, 0.62),
    '코지밀':        (0.22, 0.47),
    '푸주옥':        (0.44, 0.45),
    '비욘드비엣남':  (0.68, 0.17),
    '프레고클럽':    (0.90, 0.28),
    '이태원화로':    (0.93, 0.47),
}

G = nx.Graph()
for u, v, w in edges:
    G.add_edge(u, v, weight=w)

# ── 2. BFS 순서 계산 ──────────────────────────────────────────────
# 각 스텝: (방문된 노드 집합, 현재 활성 엣지, 방문 순서 리스트, 누적 트리 엣지 집합)
def bfs_steps(graph, start):
    visited = set([start])
    queue = deque([start])
    order = [start]
    tree_edges = set()   # BFS가 실제로 타고 온 엣지만 누적
    steps = []
    steps.append((frozenset(visited), None, list(order), frozenset(tree_edges)))

    while queue:
        node = queue.popleft()
        for neighbor in sorted(graph.neighbors(node)):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)
                order.append(neighbor)
                tree_edges.add((node, neighbor))   # 이 엣지로 이동했음을 기록
                steps.append((frozenset(visited), (node, neighbor), list(order), frozenset(tree_edges)))

    return steps

start_node = '소테'
steps = bfs_steps(G, start_node)

# ── 3. 색상 ───────────────────────────────────────────────────────
COLOR_DEFAULT   = '#EEEDFE'
COLOR_VISITED   = '#534AB7'
COLOR_CURRENT   = '#EF9F27'
COLOR_EDGE_ON   = '#EF9F27'
COLOR_EDGE_DONE = '#534AB7'
COLOR_EDGE_OFF  = '#CCCCCC'
OMONJA_COLOR    = '#F0997B'

edge_list   = list(G.edges())
edge_weight = {(u,v): d['weight'] for u,v,d in G.edges(data=True)}
edge_weight.update({(v,u): d['weight'] for u,v,d in G.edges(data=True)})

# ── 4. Figure & update 함수 ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 9))
fig.patch.set_facecolor('#F8F7FF')

def update(frame):
    ax.clear()
    ax.set_facecolor('#F8F7FF')
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.axis('off')

    visited_set, active_edge, order_list, tree_edges = steps[frame]
    current_node = order_list[-1]

    # 엣지 그리기
    for u, v in edge_list:
        x0, y0 = pos[u]; x1, y1 = pos[v]
        is_active = active_edge and {u,v} == {active_edge[0], active_edge[1]}
        # BFS 트리 엣지(실제 이동 경로)인지 확인
        is_done   = (u,v) in tree_edges or (v,u) in tree_edges

        if is_active:
            color, lw, zorder = COLOR_EDGE_ON, 3.5, 4
        elif is_done:
            color, lw, zorder = COLOR_EDGE_DONE, 2.2, 3
        else:
            color, lw, zorder = COLOR_EDGE_OFF, 1.0, 1

        ax.plot([x0,x1],[y0,y1], color=color, lw=lw, zorder=zorder, solid_capstyle='round')

        mx,my = (x0+x1)/2,(y0+y1)/2
        dx,dy = x1-x0,y1-y0
        ln = (dx**2+dy**2)**0.5 or 1
        ox,oy = -dy/ln*0.025, dx/ln*0.025
        fc = color if (is_active or is_done) else '#AAAAAA'
        ax.text(mx+ox, my+oy, str(edge_weight[(u,v)])+'분',
                fontsize=7.5, ha='center', va='center', color=fc,
                fontfamily='Malgun Gothic', zorder=5,
                bbox=dict(boxstyle='round,pad=0.15', fc='#F8F7FF', ec='none', alpha=0.85))

    # 노드 그리기
    for node in G.nodes():
        x,y = pos[node]
        is_cur = node == current_node
        is_vis = node in visited_set
        is_om  = node.startswith('오몬자')
        if is_cur:
            fc,ec,lw,r = COLOR_CURRENT,'#BA7517',2.5,0.052
        elif is_vis:
            fc,ec,lw,r = COLOR_VISITED,'#3C3489',1.5,0.046
        else:
            fc = OMONJA_COLOR if is_om else COLOR_DEFAULT
            ec = '#993C1D' if is_om else '#AFA9EC'
            lw,r = 1.0,0.046
        ax.add_patch(plt.Circle((x,y),r,fc=fc,ec=ec,lw=lw,zorder=6))
        tc = 'white' if (is_vis or is_cur) else ('#993C1D' if is_om else '#3C3489')
        ax.text(x,y,node,ha='center',va='center',fontsize=7.2,
                fontfamily='Malgun Gothic',color=tc,fontweight='bold',zorder=7)
        if is_vis:
            idx = order_list.index(node)+1
            ax.text(x+r*0.72,y+r*0.72,str(idx),fontsize=7,color='white',
                    fontweight='bold',ha='center',va='center',zorder=8,
                    fontfamily='Malgun Gothic',
                    bbox=dict(boxstyle='circle,pad=0.15',
                              fc=COLOR_CURRENT if is_cur else COLOR_VISITED,ec='none'))

    ax.text(0.5,1.03,f'BFS 탐색 순서  ({frame+1}/{len(steps)})',
            ha='center',va='center',fontsize=10,
            fontfamily='Malgun Gothic',color='#3C3489',fontweight='bold')
    ax.text(0.5,0.98,' → '.join(order_list),
            ha='center',va='center',fontsize=8,
            fontfamily='Malgun Gothic',color='#534AB7')
    ax.legend(handles=[
        mpatches.Patch(fc=COLOR_CURRENT,ec='#BA7517',label='현재 방문 중'),
        mpatches.Patch(fc=COLOR_VISITED,ec='#3C3489',label='방문 완료'),
        mpatches.Patch(fc=COLOR_DEFAULT,ec='#AFA9EC',label='미방문'),
    ], loc='lower right', prop={'family':'Malgun Gothic','size':8},
       framealpha=0.85, edgecolor='#CCCCCC')

ani = animation.FuncAnimation(fig, update, frames=len(steps), interval=1000, repeat=False)

# ── 5. mp4 저장 ───────────────────────────────────────────────────
ani.save('bfs.mp4', writer='ffmpeg', fps=1, dpi=150)
print('bfs.mp4 저장 완료!')
plt.close(fig)

bfs.mp4 저장 완료!


In [3]:
#prim

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as mpatches
import networkx as nx
import heapq

# ── 1. 그래프 데이터 ──────────────────────────────────────────────
edges = [
    ('소테',          '유림닭도리탕',  37),
    ('유림닭도리탕',  '맛있는 칼국수', 14),
    ('맛있는 칼국수', '오몬자(마포)',  11),
    ('오몬자(마포)',  '남산광어',      37),
    ('남산광어',      '와이인',        29),
    ('와이인',        '효뜨',          21),
    ('효뜨',          '문화식당',      36),
    ('문화식당',      '스페이스',      22),
    ('스페이스',      '르부아',        18),
    ('르부아',        '오몬자(강남)',  37),
    ('오몬자(강남)',  '라라면가',      39),
    ('라라면가',      '타코스퀘어',    25),
    ('타코스퀘어',    '미엔아이',      26),
    ('미엔아이',      '따로집',        39),
    ('따로집',        '코지밀',        42),
    ('코지밀',        '푸주옥',        36),
    ('푸주옥',        '비욘드비엣남',  19),
    ('비욘드비엣남',  '프레고클럽',    27),
    ('프레고클럽',    '이태원화로',    25),
    ('이태원화로',    '소테',          31),
]

pos = {
    '소테':          (0.50, 0.92),
    '유림닭도리탕':  (0.82, 0.83),
    '맛있는 칼국수': (0.94, 0.65),
    '오몬자(마포)':  (0.82, 0.45),
    '남산광어':      (0.63, 0.30),
    '와이인':        (0.38, 0.22),
    '효뜨':          (0.15, 0.35),
    '문화식당':      (0.07, 0.55),
    '스페이스':      (0.16, 0.75),
    '르부아':        (0.37, 0.90),
    '오몬자(강남)':  (0.57, 0.62),
    '라라면가':      (0.75, 0.62),
    '타코스퀘어':    (0.70, 0.78),
    '미엔아이':      (0.49, 0.76),
    '따로집':        (0.29, 0.62),
    '코지밀':        (0.22, 0.47),
    '푸주옥':        (0.44, 0.45),
    '비욘드비엣남':  (0.68, 0.17),
    '프레고클럽':    (0.90, 0.28),
    '이태원화로':    (0.93, 0.47),
}

G = nx.Graph()
for u, v, w in edges:
    G.add_edge(u, v, weight=w)

# ── 2. 프림 알고리즘 스텝 계산 ───────────────────────────────────
# 각 스텝: (MST 노드 집합, MST 엣지 집합, 현재 선택 엣지, 후보 엣지 집합, 누적 가중치)
def prim_steps(graph, start):
    steps = []
    in_mst = set([start])
    mst_edges = set()      # 확정된 MST 엣지
    total_weight = 0

    heap = []
    for nb in graph.neighbors(start):
        heapq.heappush(heap, (graph[start][nb]['weight'], start, nb))

    # 후보 엣지: 현재 힙에 있는 엣지 중 아직 MST 밖 노드로 향하는 것
    candidates = frozenset((u,v) for w,u,v in heap if v not in in_mst)
    steps.append((frozenset(in_mst), frozenset(mst_edges), None, candidates, total_weight))

    while heap and len(in_mst) < len(graph.nodes):
        w, u, v = heapq.heappop(heap)
        if v in in_mst:
            continue
        in_mst.add(v)
        mst_edges.add((u, v))
        total_weight += w
        for nb in graph.neighbors(v):
            if nb not in in_mst:
                heapq.heappush(heap, (graph[v][nb]['weight'], v, nb))
        candidates = frozenset((a,b) for wt,a,b in heap if b not in in_mst)
        steps.append((frozenset(in_mst), frozenset(mst_edges), (u, v, w), candidates, total_weight))

    return steps

start_node = '소테'
steps = prim_steps(G, start_node)

# ── 3. 색상 ───────────────────────────────────────────────────────
COLOR_DEFAULT    = '#EEEDFE'
COLOR_IN_MST     = '#534AB7'
COLOR_CURRENT    = '#EF9F27'
COLOR_CANDIDATE  = '#B8F0C8'
COLOR_EDGE_MST   = '#534AB7'
COLOR_EDGE_ON    = '#EF9F27'
COLOR_EDGE_CAND  = '#5DBB7A'
COLOR_EDGE_OFF   = '#CCCCCC'
OMONJA_COLOR     = '#F0997B'

edge_list   = list(G.edges())
edge_weight = {(u,v): d['weight'] for u,v,d in G.edges(data=True)}
edge_weight.update({(v,u): d['weight'] for u,v,d in G.edges(data=True)})

# ── 4. Figure & update 함수 ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 9))
fig.patch.set_facecolor('#F8F7FF')

def update(frame):
    ax.clear()
    ax.set_facecolor('#F8F7FF')
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.axis('off')

    in_mst, mst_edges, active, candidates, total_w = steps[frame]
    active_pair = (active[0], active[1]) if active else None
    newly_added = active[1] if active else None

    # 엣지 그리기
    for u, v in edge_list:
        x0, y0 = pos[u]; x1, y1 = pos[v]
        is_active = active_pair and {u,v} == {active_pair[0], active_pair[1]}
        is_mst    = (u,v) in mst_edges or (v,u) in mst_edges
        is_cand   = (u,v) in candidates or (v,u) in candidates

        if is_active:
            color, lw, zorder = COLOR_EDGE_ON, 3.5, 4
        elif is_mst:
            color, lw, zorder = COLOR_EDGE_MST, 2.5, 3
        elif is_cand:
            color, lw, zorder = COLOR_EDGE_CAND, 1.8, 2
        else:
            color, lw, zorder = COLOR_EDGE_OFF, 1.0, 1

        ax.plot([x0,x1],[y0,y1], color=color, lw=lw, zorder=zorder, solid_capstyle='round')

        mx,my = (x0+x1)/2,(y0+y1)/2
        dx,dy = x1-x0,y1-y0
        ln = (dx**2+dy**2)**0.5 or 1
        ox,oy = -dy/ln*0.025, dx/ln*0.025
        fc = color if (is_active or is_mst or is_cand) else '#AAAAAA'
        ax.text(mx+ox, my+oy, str(edge_weight[(u,v)])+'분',
                fontsize=7.5, ha='center', va='center', color=fc,
                fontfamily='Malgun Gothic', zorder=5,
                bbox=dict(boxstyle='round,pad=0.15', fc='#F8F7FF', ec='none', alpha=0.85))

    # 노드 그리기
    for node in G.nodes():
        x,y = pos[node]
        is_new  = node == newly_added
        is_in   = node in in_mst
        is_om   = node.startswith('오몬자')
        is_cand_node = any(node in (u,v) for u,v in candidates) and not is_in

        if is_new:
            fc,ec,lw,r = COLOR_CURRENT,'#BA7517',2.5,0.052
        elif is_in:
            fc,ec,lw,r = COLOR_IN_MST,'#3C3489',1.5,0.046
        elif is_cand_node:
            fc,ec,lw,r = COLOR_CANDIDATE,'#3A9A5C',1.2,0.046
        else:
            fc = OMONJA_COLOR if is_om else COLOR_DEFAULT
            ec = '#993C1D' if is_om else '#AFA9EC'
            lw,r = 1.0,0.046

        ax.add_patch(plt.Circle((x,y),r,fc=fc,ec=ec,lw=lw,zorder=6))
        tc = ('white' if (is_new or is_in)
              else ('#3A9A5C' if is_cand_node
              else ('#993C1D' if is_om else '#3C3489')))
        ax.text(x,y,node,ha='center',va='center',fontsize=7.2,
                fontfamily='Malgun Gothic',color=tc,fontweight='bold',zorder=7)

    # 타이틀 & 정보
    ax.text(0.5,1.03,f'프림 알고리즘 MST  ({frame+1}/{len(steps)})',
            ha='center',va='center',fontsize=10,
            fontfamily='Malgun Gothic',color='#3C3489',fontweight='bold')
    info = f'MST 엣지: {len(mst_edges)}개  |  누적 이동 시간: {total_w}분'
    if active:
        info += f'  |  선택: {active[0]} → {active[1]} ({active[2]}분)'
    ax.text(0.5,0.98,info,ha='center',va='center',fontsize=8,
            fontfamily='Malgun Gothic',color='#534AB7')

    ax.legend(handles=[
        mpatches.Patch(fc=COLOR_CURRENT,  ec='#BA7517', label='현재 추가된 노드'),
        mpatches.Patch(fc=COLOR_IN_MST,   ec='#3C3489', label='MST 확정'),
        mpatches.Patch(fc=COLOR_CANDIDATE,ec='#3A9A5C', label='후보 노드'),
        mpatches.Patch(fc=COLOR_DEFAULT,  ec='#AFA9EC', label='미방문'),
    ], loc='lower right', prop={'family':'Malgun Gothic','size':8},
       framealpha=0.85, edgecolor='#CCCCCC')

ani = animation.FuncAnimation(fig, update, frames=len(steps), interval=1200, repeat=False)

# ── 5. mp4 저장 ───────────────────────────────────────────────────
ani.save('prim.mp4', writer='ffmpeg', fps=1, dpi=150)
print('prim.mp4 저장 완료!')
plt.close(fig)

prim.mp4 저장 완료!


In [4]:
#kruskal

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as mpatches
import networkx as nx

# ── 1. 그래프 데이터 ──────────────────────────────────────────────
edges = [
    ('소테',          '유림닭도리탕',  37),
    ('유림닭도리탕',  '맛있는 칼국수', 14),
    ('맛있는 칼국수', '오몬자(마포)',  11),
    ('오몬자(마포)',  '남산광어',      37),
    ('남산광어',      '와이인',        29),
    ('와이인',        '효뜨',          21),
    ('효뜨',          '문화식당',      36),
    ('문화식당',      '스페이스',      22),
    ('스페이스',      '르부아',        18),
    ('르부아',        '오몬자(강남)',  37),
    ('오몬자(강남)',  '라라면가',      39),
    ('라라면가',      '타코스퀘어',    25),
    ('타코스퀘어',    '미엔아이',      26),
    ('미엔아이',      '따로집',        39),
    ('따로집',        '코지밀',        42),
    ('코지밀',        '푸주옥',        36),
    ('푸주옥',        '비욘드비엣남',  19),
    ('비욘드비엣남',  '프레고클럽',    27),
    ('프레고클럽',    '이태원화로',    25),
    ('이태원화로',    '소테',          31),
]

pos = {
    '소테':          (0.50, 0.92),
    '유림닭도리탕':  (0.82, 0.83),
    '맛있는 칼국수': (0.94, 0.65),
    '오몬자(마포)':  (0.82, 0.45),
    '남산광어':      (0.63, 0.30),
    '와이인':        (0.38, 0.22),
    '효뜨':          (0.15, 0.35),
    '문화식당':      (0.07, 0.55),
    '스페이스':      (0.16, 0.75),
    '르부아':        (0.37, 0.90),
    '오몬자(강남)':  (0.57, 0.62),
    '라라면가':      (0.75, 0.62),
    '타코스퀘어':    (0.70, 0.78),
    '미엔아이':      (0.49, 0.76),
    '따로집':        (0.29, 0.62),
    '코지밀':        (0.22, 0.47),
    '푸주옥':        (0.44, 0.45),
    '비욘드비엣남':  (0.68, 0.17),
    '프레고클럽':    (0.90, 0.28),
    '이태원화로':    (0.93, 0.47),
}

G = nx.Graph()
for u, v, w in edges:
    G.add_edge(u, v, weight=w)

# ── 2. Union-Find ─────────────────────────────────────────────────
class UnionFind:
    def __init__(self, nodes):
        self.parent = {n: n for n in nodes}
        self.rank   = {n: 0 for n in nodes}
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx == ry: return False
        if self.rank[rx] < self.rank[ry]: rx, ry = ry, rx
        self.parent[ry] = rx
        if self.rank[rx] == self.rank[ry]: self.rank[rx] += 1
        return True

# ── 3. 크루스칼 스텝 계산 (초기 프레임 없이 바로 첫 엣지부터) ───
# 각 스텝: (MST 엣지 집합, 현재 검토 엣지, 채택 여부, 누적 가중치, 현재 인덱스)
def kruskal_steps(graph):
    sorted_edges = sorted(graph.edges(data=True), key=lambda x: x[2]['weight'])
    uf = UnionFind(graph.nodes())
    mst_edges = set()
    total_weight = 0
    steps = []

    for idx, (u, v, d) in enumerate(sorted_edges):
        w = d['weight']
        accepted = uf.union(u, v)
        if accepted:
            mst_edges.add((u, v))
            total_weight += w
        steps.append((frozenset(mst_edges), (u, v, w), accepted, total_weight, idx))

    return steps

steps = kruskal_steps(G)
sorted_edge_list = sorted(G.edges(data=True), key=lambda x: x[2]['weight'])
sorted_edge_list = [(u,v,d['weight']) for u,v,d in sorted_edge_list]

# ── 4. 색상 ───────────────────────────────────────────────────────
COLOR_DEFAULT  = '#EEEDFE'
COLOR_IN_MST   = '#534AB7'
COLOR_ACCEPTED = '#EF9F27'
COLOR_REJECTED = '#E05C5C'
COLOR_EDGE_MST = '#534AB7'
COLOR_EDGE_OFF = '#CCCCCC'
OMONJA_COLOR   = '#F0997B'

edge_list   = list(G.edges())
edge_weight = {(u,v): d['weight'] for u,v,d in G.edges(data=True)}
edge_weight.update({(v,u): d['weight'] for u,v,d in G.edges(data=True)})

# ── 5. Figure & update 함수 ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 9))
fig.patch.set_facecolor('#F8F7FF')

def update(frame):
    ax.clear()
    ax.set_facecolor('#F8F7FF')
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.axis('off')

    mst_edges, active, accepted, total_w, cur_idx = steps[frame]
    active_pair = (active[0], active[1])

    # 엣지 그리기
    for u, v in edge_list:
        x0, y0 = pos[u]; x1, y1 = pos[v]
        is_active = {u,v} == {active_pair[0], active_pair[1]}
        is_mst    = (u,v) in mst_edges or (v,u) in mst_edges

        if is_active and accepted:
            color, lw, zorder = COLOR_ACCEPTED, 3.5, 4
        elif is_active and not accepted:
            color, lw, zorder = COLOR_REJECTED, 3.5, 4
        elif is_mst:
            color, lw, zorder = COLOR_EDGE_MST, 2.5, 3
        else:
            color, lw, zorder = COLOR_EDGE_OFF, 1.0, 1

        ax.plot([x0,x1],[y0,y1], color=color, lw=lw, zorder=zorder, solid_capstyle='round')

        mx,my = (x0+x1)/2,(y0+y1)/2
        dx,dy = x1-x0,y1-y0
        ln = (dx**2+dy**2)**0.5 or 1
        ox,oy = -dy/ln*0.025, dx/ln*0.025
        fc = color if (is_active or is_mst) else '#AAAAAA'
        ax.text(mx+ox, my+oy, str(edge_weight[(u,v)])+'분',
                fontsize=7.5, ha='center', va='center', color=fc,
                fontfamily='Malgun Gothic', zorder=5,
                bbox=dict(boxstyle='round,pad=0.15', fc='#F8F7FF', ec='none', alpha=0.85))

    # 노드 그리기
    mst_nodes = set()
    for u,v in mst_edges:
        mst_nodes.add(u); mst_nodes.add(v)
    active_nodes = {active_pair[0], active_pair[1]}

    for node in G.nodes():
        x,y = pos[node]
        is_active_node = node in active_nodes
        is_in = node in mst_nodes
        is_om = node.startswith('오몬자')

        if is_active_node and accepted:
            fc,ec,lw,r = COLOR_ACCEPTED,'#BA7517',2.5,0.052
        elif is_active_node and not accepted:
            fc,ec,lw,r = COLOR_REJECTED,'#A03030',2.5,0.052
        elif is_in:
            fc,ec,lw,r = COLOR_IN_MST,'#3C3489',1.5,0.046
        else:
            fc = OMONJA_COLOR if is_om else COLOR_DEFAULT
            ec = '#993C1D' if is_om else '#AFA9EC'
            lw,r = 1.0,0.046

        ax.add_patch(plt.Circle((x,y),r,fc=fc,ec=ec,lw=lw,zorder=6))
        tc = 'white' if (is_active_node or is_in) else ('#993C1D' if is_om else '#3C3489')
        ax.text(x,y,node,ha='center',va='center',fontsize=7.2,
                fontfamily='Malgun Gothic',color=tc,fontweight='bold',zorder=7)

    # 좌측 엣지 목록
    list_x, list_y = 0.01, 0.48
    ax.text(list_x, list_y+0.015, '── 가중치 정렬 순서 ──',
            fontsize=7, fontfamily='Malgun Gothic', color='#555', zorder=8)
    for i, (u, v, w) in enumerate(sorted_edge_list):
        y_pos = list_y - i * 0.024
        if y_pos < 0.01: break
        is_cur       = (i == cur_idx)
        is_done_edge = any((u,v)==(a,b) or (v,u)==(a,b) for a,b in mst_edges)
        color = (COLOR_ACCEPTED if (is_cur and accepted)
                 else (COLOR_REJECTED if (is_cur and not accepted)
                 else (COLOR_EDGE_MST if is_done_edge else '#AAAAAA')))
        prefix = '✓ ' if (is_done_edge and not is_cur) else ('→ ' if is_cur else '   ')
        ax.text(list_x, y_pos, f'{prefix}{u} - {v}: {w}분',
                fontsize=6.8, fontfamily='Malgun Gothic', color=color,
                fontweight='bold' if is_cur else 'normal', zorder=8)

    # 타이틀 & 정보 (1/20 ~ 20/20)
    ax.text(0.5,1.03,f'크루스칼 알고리즘 MST  ({frame+1}/{len(steps)})',
            ha='center',va='center',fontsize=10,
            fontfamily='Malgun Gothic',color='#3C3489',fontweight='bold')
    status = '채택 ✓' if accepted else '기각 ✗ (사이클)'
    info = f'MST 엣지: {len(mst_edges)}개  |  누적 이동 시간: {total_w}분  |  {active[0]} - {active[1]} ({active[2]}분): {status}'
    ax.text(0.5,0.98,info,ha='center',va='center',fontsize=8,
            fontfamily='Malgun Gothic',color='#534AB7')

    ax.legend(handles=[
        mpatches.Patch(fc=COLOR_ACCEPTED,ec='#BA7517',label='채택 (MST 추가)'),
        mpatches.Patch(fc=COLOR_REJECTED,ec='#A03030',label='기각 (사이클 형성)'),
        mpatches.Patch(fc=COLOR_IN_MST,  ec='#3C3489',label='MST 확정'),
        mpatches.Patch(fc=COLOR_DEFAULT, ec='#AFA9EC',label='미포함'),
    ], loc='lower right', prop={'family':'Malgun Gothic','size':8},
       framealpha=0.85, edgecolor='#CCCCCC')

ani = animation.FuncAnimation(fig, update, frames=len(steps), interval=1200, repeat=False)

# ── 6. mp4 저장 ───────────────────────────────────────────────────
ani.save('kruskal.mp4', writer='ffmpeg', fps=1, dpi=150)
print('kruskal.mp4 저장 완료!')
plt.close(fig)

C:\Users\안예원\AppData\Local\Temp\ipykernel_9452\2807711062.py:221: UserWarning: Glyph 10003 (\N{CHECK MARK}) missing from font(s) Malgun Gothic.
  ani.save('kruskal.mp4', writer='ffmpeg', fps=1, dpi=150)
C:\Users\안예원\AppData\Local\Temp\ipykernel_9452\2807711062.py:221: UserWarning: Glyph 10007 (\N{BALLOT X}) missing from font(s) Malgun Gothic.
  ani.save('kruskal.mp4', writer='ffmpeg', fps=1, dpi=150)


kruskal.mp4 저장 완료!
